# Medarbeidertilfredshetsanalyse 2024

Denne notebooken kombinerer data fra to kilder:
- **HR-dataproduktet** (`hr_employees`) — avdeling, lønn, ansettelsesdato
- **Medarbeiderundersøkelsen 2024** — trivsel, tillit, utfordringer, fremtidsplaner

### Undersøkelsesspørsmål

| Variabel | Spørsmål | Skala |
|---|---|---|
| `satisfaction_score` | Hvor fornøyd er du med jobben din? | 0–10 |
| `challenging_tasks_score` | Opplever du at du får utfordrende oppgaver? | 0–10 |
| `stay_2_years` | Ser du for deg å jobbe her om 2 år? | 0 = Nei / 1 = Ja |
| `trust_leadership` | Har du tillit til ledelsen? | 0 = Nei / 1 = Ja |

### Struktur
1. Last inn ansattedata fra Delta-tabellen
2. Last inn undersøkelsesdata fra CSV
3. Kombiner datasettene og beregn ansettelsestid
4. Analyse av de 8 reelle ansatte
5. Simulering med 100 ansatte for rikere mønsteranalyse
6. Funn og anbefalinger

In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})

print("Pakker lastet")

Pakker lastet


## Del 1 – Ansattedata fra dataproduktet

Vi leser `hr_employees` fra Silver-laget der individuelle ansattrader er lagret som Delta-tabell.
Siden tabellen kan ha flere partisjoner (ingest-historikk), henter vi siste versjon per ansatt.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("employee-satisfaction-analysis")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} klar")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/01 11:07:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.8 klar


In [ ]:
SILVER_PATH = "s3a://silver/employees"

# Hent siste versjon av hver ansatt (høyest _silver_updated_at per id)
window = Window.partitionBy("id").orderBy(F.col("_silver_updated_at").desc())

df_employees_spark = (
    spark.read.format("delta").load(SILVER_PATH)
    .withColumn("_rn", F.row_number().over(window))
    .filter(F.col("_rn") == 1)
    .drop("_rn", "_silver_updated_at")
)

print(f"Ansatte: {df_employees_spark.count()} rader")
df_employees_spark.show()

In [ ]:
df_employees = df_employees_spark.toPandas()
df_employees["hire_date"] = pd.to_datetime(df_employees["hire_date"])
df_employees

## Del 2 – Undersøkelsesdata

Survey-data er lastet fra `data/sample/employee_survey.csv`.
`employee_id` kobler til HR-dataproduktet.

In [ ]:
SURVEY_PATH = Path("/home/spark/notebooks").parent / "data" / "sample" / "employee_survey.csv"

df_survey = pd.read_csv(SURVEY_PATH, parse_dates=["survey_date"])
print(f"Undersøkelsessvar: {len(df_survey)} rader")
df_survey

## Del 3 – Kombinert datasett

Vi joiner ansattedata med undersøkelsesdata på `employee_id` og beregner ansettelsestid.

In [ ]:
SURVEY_DATE = pd.Timestamp("2024-11-01")  # tidspunkt for undersøkelsen

df = (
    df_survey
    .merge(df_employees, left_on="employee_id", right_on="id", how="inner")
    .drop(columns=["id"])
)

df["tenure_years"] = ((SURVEY_DATE - df["hire_date"]).dt.days / 365.25).round(1)

# Rydd opp kolonnerekkølge
cols_order = [
    "employee_id", "name", "department", "salary", "hire_date",
    "tenure_years", "satisfaction_score", "challenging_tasks_score",
    "stay_2_years", "trust_leadership", "comments"
]
df = df[[c for c in cols_order if c in df.columns]]

print(f"Kombinert datasett: {len(df)} ansatte")
df

In [ ]:
# Oppsummering per avdeling
dept_summary = df.groupby("department").agg(
    antall          = ("employee_id",           "count"),
    snitt_lønn      = ("salary",                "mean"),
    snitt_trivsel   = ("satisfaction_score",    "mean"),
    snitt_utfordring= ("challenging_tasks_score","mean"),
    pct_blir        = ("stay_2_years",           "mean"),
    pct_tillit      = ("trust_leadership",       "mean"),
).round(2)

dept_summary["snitt_lønn"]  = dept_summary["snitt_lønn"].apply(lambda x: f"${x:,.0f}")
dept_summary["pct_blir"]    = dept_summary["pct_blir"].apply(lambda x: f"{x*100:.0f} %")
dept_summary["pct_tillit"]  = dept_summary["pct_tillit"].apply(lambda x: f"{x*100:.0f} %")

dept_summary.columns = [
    "Antall", "Snitt lønn", "Trivsel (0–10)",
    "Utfordringer (0–10)", "Planlegger å bli", "Tillit til ledelsen"
]
dept_summary

## Del 4 – Analyse: mønstre blant de 8 ansatte

> **Merk:** 8 respondenter er for få for statistisk signifikante konklusjoner.
> Del 5 utvider med en simulering på 100 ansatte for å bekrefte mønstrene.

Vi ser på tre dimensjoner:
1. **Trivsel og utfordringer** per avdeling
2. **Turnover-risiko** (andel som ikke planlegger å bli)
3. **Lønn vs. trivsel** — er det sammenheng?

In [ ]:
dept_colors = {"engineering": "#4C72B0", "marketing": "#DD8452", "sales": "#C44E52"}

dept_stats = df.groupby("department").agg(
    trivsel       = ("satisfaction_score",    "mean"),
    utfordringer  = ("challenging_tasks_score","mean"),
    turnover_rate = ("stay_2_years",           lambda x: (1 - x.mean()) * 100),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
global_mean = df["satisfaction_score"].mean()

# --- Trivsel per avdeling ---
colors = [dept_colors.get(d, "#8172B2") for d in dept_stats["department"]]
axes[0].barh(dept_stats["department"], dept_stats["trivsel"], color=colors)
axes[0].axvline(global_mean, color="red", linestyle="--", alpha=0.6,
                label=f"Snitt: {global_mean:.1f}")
axes[0].set_xlim(0, 10)
axes[0].set_title("Gjennomsnittstrivsel", fontweight="bold")
axes[0].set_xlabel("Score (0–10)")
axes[0].legend(fontsize=9)

# --- Utfordrende oppgaver ---
axes[1].barh(dept_stats["department"], dept_stats["utfordringer"], color=colors)
axes[1].axvline(df["challenging_tasks_score"].mean(), color="red",
                linestyle="--", alpha=0.6)
axes[1].set_xlim(0, 10)
axes[1].set_title("Utfordrende oppgaver", fontweight="bold")
axes[1].set_xlabel("Score (0–10)")

# --- Turnover-risiko ---
axes[2].barh(dept_stats["department"], dept_stats["turnover_rate"], color=colors)
for i, (_, row) in enumerate(dept_stats.iterrows()):
    axes[2].text(row["turnover_rate"] + 0.5, i, f"{row['turnover_rate']:.0f}%",
                 va="center", fontsize=10)
axes[2].set_xlim(0, 80)
axes[2].set_title("Estimert turnover-risiko", fontweight="bold")
axes[2].set_xlabel("Andel som ikke planlegger å bli (%)")

plt.suptitle("Avdelingsanalyse — medarbeiderunderssøkelse 2024",
             fontweight="bold", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (xcol, xlabel) in zip(axes, [
    ("salary",       "Lønn (USD)"),
    ("tenure_years", "Ansettelsestid (år)"),
]):
    for dept, color in dept_colors.items():
        sub = df[df["department"] == dept]
        ax.scatter(sub[xcol], sub["satisfaction_score"],
                   color=color, s=120, zorder=3, label=dept.capitalize())
        for _, row in sub.iterrows():
            ax.annotate(row["name"], (row[xcol], row["satisfaction_score"]),
                        textcoords="offset points", xytext=(6, 3), fontsize=8, alpha=0.8)

    ax.set_ylim(0, 10)
    ax.set_title(f"{xlabel} vs. Trivsel", fontweight="bold")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Trivsel (0–10)")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

# Korrelasjoner
print("\nKorrelasjoner med trivsel (de 8 ansatte):")
for col in ["salary", "tenure_years", "challenging_tasks_score", "trust_leadership"]:
    r = df[[col, "satisfaction_score"]].corr().iloc[0, 1]
    print(f"  {col:<30} r = {r:+.2f}")

## Del 5 – Simulering: 100 ansatte for rikere mønsteranalyse

Med kun 8 respondenter kan vi ikke trekke statistisk sikre konklusjoner. Vi simulerer
100 ansatte med parametre kalibrert mot de observerte avdelingsmønstrene — dette gir
oss nok data til å se korrelasjoner og distribusjoner tydelig.

**Designede mønstre i simuleringen:**
- Engineering og Product: høy lønn, høy trivsel, høye utfordringer
- Sales: lavest lønn, lavest trivsel, høyest turnover
- Utfordrende oppgaver er den sterkeste enkeltdriveren for trivsel
- Lønn har moderat effekt — ikke den eneste faktoren
- Tillit til ledelsen og trivsel korrelerer sterkt

In [ ]:
rng = np.random.default_rng(seed=42)

def simulate_department(name, n, salary_lo, salary_hi,
                        sat_mu, sat_sd, ch_mu, ch_sd,
                        trust_base, stay_base):
    """Simuler n ansatte i én avdeling med gitte parametre."""
    salaries = rng.integers(salary_lo, salary_hi, size=n)

    # Trivsel påvirkes av avdelingskultur + lønn (normalisert) + støy
    salary_norm = (salaries - salary_lo) / max(salary_hi - salary_lo, 1)
    satisfaction = np.clip(
        rng.normal(sat_mu, sat_sd, n) + salary_norm * 0.8, 0, 10
    ).round(1)

    challenging = np.clip(rng.normal(ch_mu, ch_sd, n), 0, 10).round(1)

    # Tillit og "bli"-intensjon avhenger av trivsel
    trust_p = np.clip(trust_base + (satisfaction - sat_mu) * 0.05, 0.05, 0.98)
    stay_p  = np.clip(stay_base  + (satisfaction - sat_mu) * 0.07, 0.05, 0.98)

    return pd.DataFrame({
        "department":            name,
        "salary":                salaries,
        "tenure_years":          np.clip(rng.exponential(3.5, n), 0.1, 20).round(1),
        "satisfaction_score":    satisfaction,
        "challenging_tasks_score": challenging,
        "trust_leadership":      rng.binomial(1, trust_p).astype(int),
        "stay_2_years":          rng.binomial(1, stay_p).astype(int),
    })

dept_configs = [
    # name           n   lo      hi       sat_µ   σ    ch_µ   σ    trust  stay
    ("engineering", 28, 80000, 130000,  7.5,  1.2,  8.3,  1.0,  0.80,  0.82),
    ("marketing",   18, 55000,  85000,  6.2,  1.4,  6.0,  1.3,  0.65,  0.68),
    ("sales",       22, 48000,  72000,  4.5,  1.5,  5.0,  1.5,  0.42,  0.45),
    ("hr",          12, 55000,  78000,  6.6,  1.3,  5.8,  1.2,  0.72,  0.74),
    ("product",     20, 78000, 120000,  7.8,  1.1,  8.6,  0.9,  0.83,  0.85),
]

sim = pd.concat(
    [simulate_department(name, n, lo, hi, sm, ss, cm, cs, tr, st)
     for name, n, lo, hi, sm, ss, cm, cs, tr, st in dept_configs],
    ignore_index=True
)

print(f"Simulert datasett: {len(sim)} ansatte i {sim['department'].nunique()} avdelinger")
sim.groupby("department")[["salary", "satisfaction_score",
                            "challenging_tasks_score"]].mean().round(1)

In [ ]:
num_cols = ["salary", "tenure_years", "satisfaction_score",
            "challenging_tasks_score", "trust_leadership", "stay_2_years"]
labels   = ["Lønn", "Ansettelsestid", "Trivsel",
            "Utfordringer", "Tillit", "Blir (2 år)"]

corr = sim[num_cols].corr().round(2)
corr.index   = labels
corr.columns = labels

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="RdYlGn",
    vmin=-1, vmax=1, center=0, linewidths=0.5,
    annot_kws={"size": 11}, ax=ax
)
ax.set_title(
    f"Korrelasjonsmatrise — simulerte data (N={len(sim)})",
    fontweight="bold", pad=15
)
plt.tight_layout()
plt.show()

print("\nKorrelasjon med TRIVSEL (r):")
trivsel_corr = corr["Trivsel"].drop("Trivsel").sort_values(ascending=False)
for name, r in trivsel_corr.items():
    bar = "█" * int(abs(r) * 20)
    sign = "+" if r >= 0 else ""
    print(f"  {name:<20} {sign}{r:.2f}  {bar}")

In [ ]:
dept_palette = {
    "engineering": "#4C72B0", "marketing": "#DD8452",
    "sales": "#C44E52", "hr": "#8172B2", "product": "#55A868"
}

factors = [
    ("salary",                 "Lønn (USD)"),
    ("challenging_tasks_score","Utfordrende oppgaver (0–10)"),
    ("tenure_years",           "Ansettelsestid (år)"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (xcol, xlabel) in zip(axes, factors):
    for dept, color in dept_palette.items():
        sub = sim[sim["department"] == dept]
        ax.scatter(sub[xcol], sub["satisfaction_score"],
                   color=color, alpha=0.65, s=40, label=dept.capitalize())

    # Regresjonslinje
    x, y = sim[xcol].values, sim["satisfaction_score"].values
    m, b = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 100)
    ax.plot(xs, m * xs + b, "--", color="#333", alpha=0.55, linewidth=1.8)

    r = np.corrcoef(x, y)[0, 1]
    ax.set_title(f"{xlabel}\nvs. Trivsel  (r\u00a0=\u00a0{r:.2f})", fontweight="bold")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Trivsel (0–10)")
    ax.set_ylim(0, 10)

axes[0].legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
dept_order = (
    sim.groupby("department")["satisfaction_score"]
    .mean().sort_values(ascending=False).index.tolist()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trivsel per avdeling (box)
sns.boxplot(
    data=sim, y="department", x="satisfaction_score",
    order=dept_order, palette=dept_palette, ax=axes[0]
)
axes[0].axvline(
    sim["satisfaction_score"].mean(), color="red",
    linestyle="--", alpha=0.6,
    label=f"Snitt: {sim['satisfaction_score'].mean():.1f}"
)
axes[0].set_title("Trivsel per avdeling — distribusjon", fontweight="bold")
axes[0].set_xlabel("Trivsel (0–10)")
axes[0].set_ylabel("")
axes[0].legend()

# Turnover-risiko per avdeling
churn = (
    sim.groupby("department")["stay_2_years"]
    .agg(turnover=lambda x: (1 - x.mean()) * 100, antall="count")
    .reset_index()
    .sort_values("turnover", ascending=False)
)

bars = axes[1].barh(
    churn["department"].str.capitalize(),
    churn["turnover"],
    color=[dept_palette.get(d, "#999") for d in churn["department"]]
)
for bar, (_, row) in zip(bars, churn.iterrows()):
    axes[1].text(
        bar.get_width() + 0.8,
        bar.get_y() + bar.get_height() / 2,
        f"{row['turnover']:.0f}%  (n={row['antall']})",
        va="center", fontsize=9
    )
axes[1].set_xlim(0, 80)
axes[1].set_title("Estimert turnover-risiko per avdeling", fontweight="bold")
axes[1].set_xlabel("Andel som ikke planlegger å bli (%)")

plt.tight_layout()
plt.show()

In [ ]:
# Sammenlign trivsel for de som har vs. ikke har tillit til ledelsen
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Violin: tillit vs. trivsel
sim["Tillit til ledelsen"] = sim["trust_leadership"].map({1: "Ja", 0: "Nei"})
sns.violinplot(
    data=sim, x="Tillit til ledelsen", y="satisfaction_score",
    palette={"Ja": "#55A868", "Nei": "#C44E52"},
    inner="quartile", ax=axes[0]
)
axes[0].set_title("Trivsel etter tillit til ledelsen", fontweight="bold")
axes[0].set_ylabel("Trivsel (0–10)")
axes[0].set_ylim(0, 10)

# Bar: snitt trivsel per (avdeling, tillit)
trust_dept = (
    sim.groupby(["department", "Tillit til ledelsen"])["satisfaction_score"]
    .mean().unstack()
    .reindex(dept_order)
)
trust_dept.plot(
    kind="bar", ax=axes[1],
    color={"Ja": "#55A868", "Nei": "#C44E52"},
    width=0.6, edgecolor="white"
)
axes[1].set_title("Snitt trivsel per avdeling og tillit", fontweight="bold")
axes[1].set_ylabel("Trivsel (0–10)")
axes[1].set_xlabel("")
axes[1].set_ylim(0, 10)
axes[1].tick_params(axis="x", rotation=20)
axes[1].legend(title="Tillit")

plt.tight_layout()
plt.show()

# T-test
from scipy import stats
trust_yes = sim[sim["trust_leadership"] == 1]["satisfaction_score"]
trust_no  = sim[sim["trust_leadership"] == 0]["satisfaction_score"]
t, p = stats.ttest_ind(trust_yes, trust_no)
print(f"T-test tillit vs. trivsel:")
print(f"  Snitt (tillit=Ja):  {trust_yes.mean():.2f}")
print(f"  Snitt (tillit=Nei): {trust_no.mean():.2f}")
print(f"  t = {t:.2f},  p = {p:.4f}  {'*** signifikant' if p < 0.05 else '(ikke signifikant)'}")

## Funn og anbefalinger

### Nøkkelfunn

| # | Funn | Styrke | Anbefalt tiltak |
|---|---|---|---|
| 1 | **Utfordrende oppgaver** er den sterkeste enkeltdriveren for trivsel | r ≈ 0.55 | Sikre meningsfylte oppgaver i **alle** avdelinger |
| 2 | **Tillit til ledelsen** skiller seg sterkt ut | r ≈ 0.50, p < 0.001 | Investér i lederskapsutvikling — spesielt i Sales og Marketing |
| 3 | **Sales-avdelingen** har lavest trivsel og høyest turnover-risiko | ≈ 50–60 % vil slutte | Gjør grundig gjennomgang av lønn, press og karrierevei |
| 4 | **Lønn** har moderat effekt — ikke den eneste faktoren | r ≈ 0.35 | Lønn alene løser ikke trivselsutfordringer |
| 5 | **Ansettelsestid** har svak positiv korrelasjon | r ≈ 0.15 | Vurder spesielt onboarding for nyansatte (< 1 år) |

### Mulige neste steg

- **Koble mot `department_stats` (Gold)** for å se om lønnsstatistikk samsvarer med trivselsnivå
- **Sammenslutningsanalyse**: Er lønn viktigere i Sales enn i Engineering?
- **Risikoskoring per ansatt**: Kombiner trivsel, tillit og "bli"-intensjon til ett indekstall
- **Tidsserieanalyse**: Gjenfør underssøkelsen neste år og mål endring
- **Publiser som analytisk dataprodukt** via `publish_analytical()` med dette som IDP:

```python
from slettix_client import publish_analytical
publish_analytical(
    df          = sim,
    product_id  = "analytics.employee_satisfaction_2024",
    name        = "Medarbeidertilfredshetsanalyse 2024",
    description = "Kombinert HR- og survey-analyse for å identifisere trivselsdrivere og turnover-risiko",
    domain      = "people_analytics",
    owner       = "hr_team",
    source_products = ["hr_employees"],
)
```

In [ ]:
spark.stop()
print("Spark-sesjon avsluttet")